# Collecte : books.toscrape.com

## Cadre légal et éthique

| Question | Réponse |
|---|---|
| 1. Une API ou un open data existent-ils ? | Non, le site ne propose ni API ni téléchargement. |
| 2. Que dit le robots.txt ? | https://books.toscrape.com/robots.txt est vide : rien n'est interdit, pas de Crawl-delay. |
| 3. Que disent les CGU ? | Pas de CGU : c'est un site d'entraînement conçu pour le scraping. |
| 4. Données personnelles ? | Non : titres, prix, notes, stocks, UPC. Aucun nom de personne collecté. |
| 5. Combien de requêtes, à quel rythme ? | Une catégorie (Historical Fiction) : 2 pages de liste + 26 fiches = 28 requêtes, 1 requête/seconde, User-Agent identifié. |
| 6. Je garde le résultat ? | Oui, en CSV dans `../data/`. |

In [1]:
import requests

URL = "https://books.toscrape.com/catalogue/page-1.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (sebastien.hetsch.data@gmail.com)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"

print("status   :", r.status_code)
print("longueur :", len(r.text), "caracteres")
print("debut    :", r.text[:60])


status   : 200
longueur : 50449 caracteres
debut    : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [2]:
print("le prix est present  :", "£51.77" in r.text)
print("le titre est present :", "A Light in the Attic" in r.text)


le prix est present  : True
le titre est present : True


In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")
fiches = soup.select("article.product_pod")
print("fiches trouvees :", len(fiches))


fiches trouvees : 20


In [4]:
fiche = fiches[0]

prix = fiche.select_one("p.price_color").get_text(strip=True)   # contenu
lien_h3 = fiche.select_one("h3 a")
titre = lien_h3["title"]                                         # attribut
href = lien_h3["href"]                                           # attribut

print("prix  :", prix)
print("titre :", titre)
print("href  :", href)


prix  : £51.77
titre : A Light in the Attic
href  : a-light-in-the-attic_1000/index.html


In [5]:
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

classes = fiche.select_one("p.star-rating")["class"]
print("classes :", classes)
print("note    :", NOTES[classes[1]])


classes : ['star-rating', 'Three']
note    : 3


In [6]:
from urllib.parse import urljoin

print(urljoin(URL, href))


https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


In [7]:
def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }

print(lire_fiche(fiches[0], URL))


{'titre': 'A Light in the Attic', 'prix': '£51.77', 'dispo': 'In stock', 'note': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


In [8]:
suivant = soup.select_one("li.next a")
print(suivant["href"] if suivant else "pas de page suivante")


page-2.html


In [9]:
import time

def collecter_toutes_les_pages(url_depart, pages_max=3):
    url = url_depart
    livres = []
    page = 0
    while url and page < pages_max:
        page += 1
        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f"  page {page} : status {r.status_code}, on s'arrete")
            break
        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f, url) for f in fiches]
        print(f"  page {page} : {len(fiches)} livres  (total {len(livres)})")
        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None
        time.sleep(1)
    return livres


In [10]:
livres = collecter_toutes_les_pages("https://books.toscrape.com/catalogue/page-1.html")
print("collecte :", len(livres), "livres")


  page 1 : 20 livres  (total 20)


  page 2 : 20 livres  (total 40)


  page 3 : 20 livres  (total 60)


collecte : 60 livres


## Partie 5 : le second niveau — lire une fiche détaillée

In [11]:
def lire_detail(url):
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200:
        return {"categorie": None, "stock": None, "upc": None}
    soup = BeautifulSoup(r.text, "html.parser")

    # le tableau : le th en cle, le td en valeur
    infos = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val:
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)

    # la categorie : dernier lien du fil d'Ariane
    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    return {
        "categorie": categorie,
        "stock": infos.get("Availability"),
        "upc": infos.get("UPC"),
    }

url_test = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
print(lire_detail(url_test))

{'categorie': 'Poetry', 'stock': 'In stock (22 available)', 'upc': 'a897fe39b1053632'}


In [12]:
import re

def nombre_en_stock(texte):
    if not texte:
        return None
    trouve = re.search(r"\((\d+) available\)", texte)
    return int(trouve.group(1)) if trouve else 0

print(nombre_en_stock("In stock (22 available)"))
print(nombre_en_stock("Out of stock"))
print(nombre_en_stock(None))

22
0
None


# Challenge : collecter une catégorie sur deux niveaux

## Étape 1 : fiche technique — catégorie *Historical Fiction*

URL de départ : https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html

**Page de liste**

| Donnée | Sélecteur | Contenu ou attribut |
|---|---|---|
| bloc répété (un livre) | `article.product_pod` | — |
| titre complet | `h3 a` | attribut `title` |
| prix | `p.price_color` | contenu |
| disponibilité | `p.instock` | contenu |
| note | `p.star-rating` | attribut `class` (2e valeur : One…Five) |
| lien vers la fiche | `h3 a` | attribut `href` (relatif → `urljoin`) |
| page suivante | `li.next a` | attribut `href` — présent sur la page 1, absent sur la page 2 |

**Fiche détaillée**

| Donnée | Sélecteur | Contenu ou attribut |
|---|---|---|
| catégorie | `ul.breadcrumb li a` (dernier lien) | contenu |
| stock | `table.table-striped tr` → `th` = *Availability* | contenu du `td` |
| UPC | `table.table-striped tr` → `th` = *UPC* | contenu du `td` |

**Nombre de livres annoncé par le site : 26** (« 26 results », sur 2 pages).

## Étape 2 : une page, pour valider la chaîne

In [13]:
import pandas as pd

URL_CATEGORIE = "https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html"

livres_p1 = collecter_toutes_les_pages(URL_CATEGORIE, pages_max=1)
df_p1 = pd.DataFrame(livres_p1)

print("titres tronques :", df_p1["titre"].str.endswith("...").sum())
print("notes trouvees  :", sorted(df_p1["note"].unique()))
df_p1

  page 1 : 20 livres  (total 20)


titres tronques : 0
notes trouvees  : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


,titre,prix,dispo,note,url
0,Tipping the Velvet,£53.74,In stock,1,https://books.toscrape.com/catalogue/tipping-t...
1,Forever and Forever: The Courtship of Henry Lo...,£29.69,In stock,3,https://books.toscrape.com/catalogue/forever-a...
2,A Flight of Arrows (The Pathfinders #2),£55.53,In stock,5,https://books.toscrape.com/catalogue/a-flight-...
3,The House by the Lake,£36.95,In stock,1,https://books.toscrape.com/catalogue/the-house...
4,Mrs. Houdini,£30.25,In stock,5,https://books.toscrape.com/catalogue/mrs-houdi...
5,The Marriage of Opposites,£28.08,In stock,4,https://books.toscrape.com/catalogue/the-marri...
6,Glory over Everything: Beyond The Kitchen House,£45.84,In stock,3,https://books.toscrape.com/catalogue/glory-ove...
7,"Love, Lies and Spies",£20.55,In stock,2,https://books.toscrape.com/catalogue/love-lies...
8,A Paris Apartment,£39.01,In stock,4,https://books.toscrape.com/catalogue/a-paris-a...
9,Lilac Girls,£17.28,In stock,2,https://books.toscrape.com/catalogue/lilac-gir...


## Étape 3 : toutes les pages de liste

In [14]:
NB_ANNONCE = 26

livres = collecter_toutes_les_pages(URL_CATEGORIE, pages_max=10)  # garde-fou : 10 pages maximum
print("livres collectes :", len(livres), "/ annonces :", NB_ANNONCE)
print("conforme :", len(livres) == NB_ANNONCE)

  page 1 : 20 livres  (total 20)


  page 2 : 6 livres  (total 26)


livres collectes : 26 / annonces : 26
conforme : True


## Étape 4 : le second niveau — visiter chaque fiche

In [15]:
details = []
for i, livre in enumerate(livres, start=1):
    try:
        details.append(lire_detail(livre["url"]))
    finally:
        time.sleep(1)  # une requete par seconde, meme en cas d'erreur
    print(f"  fiche {i}/{len(livres)} : {details[-1]['categorie']} - {details[-1]['stock']}")

  fiche 1/26 : Historical Fiction - In stock (20 available)


  fiche 2/26 : Historical Fiction - In stock (16 available)


  fiche 3/26 : Historical Fiction - In stock (16 available)


  fiche 4/26 : Historical Fiction - In stock (15 available)


  fiche 5/26 : Historical Fiction - In stock (15 available)


  fiche 6/26 : Historical Fiction - In stock (14 available)


... (16 outputs left out)

  fiche 23/26 : Historical Fiction - In stock (1 available)


  fiche 24/26 : Historical Fiction - In stock (1 available)


  fiche 25/26 : Historical Fiction - In stock (1 available)


  fiche 26/26 : Historical Fiction - In stock (1 available)


## Étape 5 : assembler et nettoyer

In [16]:
df = pd.DataFrame(livres).join(pd.DataFrame(details))

df["prix_gbp"] = df["prix"].str.replace("£", "", regex=False).astype(float)
df["stock"] = df["stock"].apply(nombre_en_stock).astype(int)
df["note"] = df["note"].astype(int)

df = df[["titre", "categorie", "prix_gbp", "note", "stock", "upc", "url"]]

print("lignes            :", len(df))
print("UPC uniques       :", df["upc"].nunique())
print("titres tronques   :", df["titre"].str.endswith("...").sum())
print("notes vides       :", df["note"].isna().sum())
print()
print(df.dtypes)
df.head()

lignes            : 26
UPC uniques       : 26
titres tronques   : 0
notes vides       : 0

titre            str
categorie        str
prix_gbp     float64
note           int64
stock          int64
upc              str
url              str
dtype: object


,titre,categorie,prix_gbp,note,stock,upc,url
0,Tipping the Velvet,Historical Fiction,53.74,1,20,90fa61229261140a,https://books.toscrape.com/catalogue/tipping-t...
1,Forever and Forever: The Courtship of Henry Lo...,Historical Fiction,29.69,3,16,9beb96a2300109d7,https://books.toscrape.com/catalogue/forever-a...
2,A Flight of Arrows (The Pathfinders #2),Historical Fiction,55.53,5,16,3a6fb983e2554023,https://books.toscrape.com/catalogue/a-flight-...
3,The House by the Lake,Historical Fiction,36.95,1,15,9287e238d7c3b06e,https://books.toscrape.com/catalogue/the-house...
4,Mrs. Houdini,Historical Fiction,30.25,5,15,a40723994f715420,https://books.toscrape.com/catalogue/mrs-houdi...


## Étape 6 : analyser

In [17]:
print("1. Nombre de livres :", len(df))
print("   Stock total      :", df["stock"].sum(), "exemplaires")
print()
print("2. Prix moyen       :", round(df["prix_gbp"].mean(), 2), "£")
plus_cher = df.loc[df["prix_gbp"].idxmax()]
print("   Livre le plus cher :", plus_cher["titre"], "-", plus_cher["prix_gbp"], "£")
print()
print("3. Repartition des notes :")
print(df["note"].value_counts().sort_index().to_string())
print()
print("4. Valeurs de la colonne categorie :", df["categorie"].unique())

1. Nombre de livres : 26
   Stock total      : 194 exemplaires

2. Prix moyen       : 33.64 £
   Livre le plus cher : The Last Painting of Sara de Vos - 55.55 £

3. Repartition des notes :
note
1    5
2    4
3    5
4    4
5    8

4. Valeurs de la colonne categorie : <StringArray>
['Historical Fiction']
Length: 1, dtype: str


### Réponses

1. **26 livres** (autant que le nombre annoncé par le site), pour un **stock total de 194 exemplaires**.
2. **Prix moyen : 33,64 £**. Le livre le plus cher est *The Last Painting of Sara de Vos* (**55,55 £**).
3. **Notes** : 1★ = 5 livres · 2★ = 4 · 3★ = 5 · 4★ = 4 · 5★ = 8. La note 5 est la plus fréquente.
4. La colonne `categorie` contient **une seule valeur, *Historical Fiction***. La collecte est donc cohérente : tous les livres visités viennent bien de la catégorie demandée. Si plusieurs valeurs étaient apparues, cela aurait signalé une erreur de périmètre (mauvaise URL de départ, ou pagination sortie de la catégorie).

## Étape 7 : livrer

In [18]:
chemin = "../data/historical_fiction.csv"
df.to_csv(chemin, index=False, encoding="utf-8-sig")
print("ecrit :", chemin, "-", len(df), "lignes")

ecrit : ../data/historical_fiction.csv - 26 lignes
